In [1]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

app_id = os.getenv("REMOVED")
app_key = os.getenv("REMOVED")

print(app_id is not None)
print(app_key is not None)

False
False


In [ ]:
import os

print(os.getcwd())
print(os.path.exists(".env"))

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

app_id = os.getenv("REMOVED")
app_key = os.getenv("REMOVED")

print(app_id is not None)
print(app_key is not None)

In [ ]:
from pathlib import Path

env_path = Path("../.env").resolve()

print(env_path)
print(env_path.exists())

In [ ]:
from dotenv import dotenv_values

env_data = dotenv_values("../.env")

print(env_data.keys())


In [ ]:
print(bool(env_data.get("ADZUNA_APP_ID")))
print(bool(env_data.get("ADZUNA_APP_KEY")))

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("../.env", override=True)

app_id = os.getenv("REMOVED")
app_key = os.getenv("REMOVED")

print(app_id is not None)
print(app_key is not None)

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path("../.env").resolve()

loaded = load_dotenv(dotenv_path=env_path, override=True)

print("File loaded:", loaded)
print("APP_ID exists:", "ADZUNA_APP_ID" in os.environ)
print("APP_KEY exists:", "ADZUNA_APP_KEY" in os.environ)

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path("../.env").resolve()
load_dotenv(dotenv_path=env_path, override=True)

app_id = os.getenv("ADZUNA_APP_ID")
app_key = os.getenv("ADZUNA_APP_KEY")

print(app_id is not None)
print(app_key is not None)

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

load_dotenv()
app_id = os.getenv("ADZUNA_APP_ID")
app_key = os.getenv("ADZUNA_APP_KEY")

url = "https://api.adzuna.com/v1/api/jobs/in/search/1"
params = {
    "app_id": app_id,
    "app_key": app_key,
    "results_per_page": 50,
    "what": "data analyst",
}

response = requests.get(url, params=params)
print(response.status_code)

if response.status_code == 200:
    data = response.json()
    os.makedirs("../data/raw", exist_ok=True)
    with open("../data/raw/adzuna_page1.json", "w", encoding="utf-8") as f:
        json.dump(data, f)
    print("Saved. Jobs in this page:", len(data["results"]))
else:
    print("Failed:", response.status_code)

In [ ]:
import os
import json

os.makedirs("../data/raw", exist_ok=True)

with open("../data/raw/adzuna_page1.json", "w", encoding="utf-8") as f:
    json.dump(data, f)

print("Saved. Jobs in this page:", len(data["results"]))

In [ ]:
import json

with open("../data/raw/adzuna_page1.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

jobs = raw["results"]
first_job = jobs[0]

print(len(jobs))
print(first_job.keys())

In [ ]:
print(first_job["title"])
print(first_job["company"])
print(first_job["location"])
print(first_job["created"])
print(first_job["salary_is_predicted"])
print(len(first_job["description"]))

In [ ]:
print(first_job["description"])

In [ ]:
print(first_job["company"]["display_name"])
print(first_job["location"]["display_name"])
print(first_job["location"]["area"][-1])

In [ ]:
for job in jobs[:5]:
    title = job["title"]
    company = job["company"],["display_name"]
    city = job["location"]["area"][-1]
    print(title, "|", company, "|", city)

In [ ]:
print(raw.keys())
print(raw.get("count"))

In [ ]:
print(raw["mean"])

In [ ]:
with_salary = 0

for job in jobs:
    if "salary_min" in job:
        with_salary = with_salary + 1

print(with_salary, "of", len(jobs))

In [ ]:
import os
import json
import time
import requests

searches = [
    ("data analyst", 2, 5),
    ("business analyst", 1, 4),
    ("data scientist", 1, 3),
    ("data engineer", 1, 3),
    ("power bi", 1, 3),
    ("machine learning", 1, 3),
]

url_base = "https://api.adzuna.com/v1/api/jobs/in/search/"
stop = False

for term, first_page, last_page in searches:
    if stop:
        break
    for page in range(first_page, last_page + 1):
        params = {
            "app_id": app_id,
            "app_key": app_key,
            "results_per_page": 50,
            "what": term,
        }
        response = requests.get(url_base + str(page), params=params)
        print(term, page, response.status_code)

        if response.status_code != 200:
            print("Stopped at", term, page)
            stop = True
            break

        name = term.replace(" ", "_")
        with open(f"../data/raw/adzuna_{name}_p{page}.json", "w", encoding="utf-8") as f:
            json.dump(response.json(), f)

        time.sleep(13)

print("Done")

In [ ]:
from pathlib import Path

raw_folder = Path("../data/raw")

print("Folder exists:", raw_folder.exists())

if raw_folder.exists():
    for file in sorted(raw_folder.glob("*.json")):
        print(file.name, "-", round(file.stat().st_size / 1024, 2), "KB")

In [ ]:
import json
from pathlib import Path

raw_folder = Path("../data/raw")

files = [
    "adzuna_page1.json",
    "adzuna_data_analyst_p2.json"
]

for filename in files:
    file_path = raw_folder / filename

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    jobs = data.get("results", [])
    print(filename, "→", len(jobs), "jobs")

In [ ]:

import json
import pandas as pd
from pathlib import Path

raw_folder = Path("../data/raw")

file_names = [
    "adzuna_page1.json",
    "adzuna_data_analyst_p2.json"
]

all_jobs = []

for file_name in file_names:
    file_path = raw_folder / file_name

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    all_jobs.extend(data.get("results", []))

df = pd.json_normalize(all_jobs)

print("Total records:", len(df))
print("Total columns:", len(df.columns))

display(df.head())


In [ ]:

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values in the top 15 columns:")
print(df.isna().sum().sort_values(ascending=False).head(15))

print("\nDuplicate records:", df.duplicated().sum())


In [ ]:

print("Total records:", len(df))
print("Unique job IDs:", df["id"].nunique())
print("Duplicate job IDs:", df["id"].duplicated().sum())

print("\nDuplicate job records:")
display(
    df[df["id"].duplicated(keep=False)]
    [["id", "title", "company.display_name", "location.display_name"]]
    .sort_values("id")
)


In [ ]:

df["description_length"] = df["description"].str.len()

print("Shortest description:", df["description_length"].min())
print("Median description length:", df["description_length"].median())
print("Longest description:", df["description_length"].max())

print("\nDescription length summary:")
display(df["description_length"].describe())

print("\nFive shortest descriptions:")
display(
    df.nsmallest(5, "description_length")
    [["title", "company.display_name", "description_length", "description"]]
)


In [ ]:

# Inspect descriptions that are exactly 500 characters
at_limit = df[df["description_length"] == 500]

print("Descriptions at 500 characters:", len(at_limit))
print("\nSample descriptions at the limit:")

for _, row in at_limit.head(3).iterrows():
    print("\nJob:", row["title"])
    print("Company:", row["company.display_name"])
    print("Description ending:", repr(row["description"][-100:]))


In [ ]:

# Check important fields before cleaning
important_columns = [
    "id",
    "title",
    "company.display_name",
    "location.display_name",
    "created",
    "description",
    "salary_min",
    "salary_max"
]

print(df[important_columns].isna().sum())


In [ ]:

# Create a separate copy for cleaning
df_clean = df.copy()

# Replace missing company names with "Unknown"
df_clean["company.display_name"] = (
    df_clean["company.display_name"].fillna("Unknown")
)

# Check the result
print("Original records:", len(df))
print("Cleaned records:", len(df_clean))

print("\nMissing company names after cleaning:")
print(df_clean["company.display_name"].isna().sum())

print("\nMissing salary values after cleaning:")
print(df_clean[["salary_min", "salary_max"]].isna().sum())


In [ ]:
# Convert the created column from text into datetime values
df_clean["created"] = pd.to_datetime(
    df_clean["created"],
    errors="coerce",
    utc=True
)

# Check whether the conversion worked
print("Date column data type:", df_clean["created"].dtype)

print("\nEarliest job posting:")
print(df_clean["created"].min())

print("\nLatest job posting:")
print(df_clean["created"].max())

print("\nInvalid dates:", df_clean["created"].isna().sum())

In [ ]:
# Select jobs that have both salary values
salary_available = df_clean[
    df_clean["salary_min"].notna()
    & df_clean["salary_max"].notna()
]

# Find records where minimum salary is greater than maximum salary
invalid_salary = salary_available[
    salary_available["salary_min"] > salary_available["salary_max"]
]

print("Jobs with both salary values:", len(salary_available))
print("Jobs with invalid salary ranges:", len(invalid_salary))

display(
    invalid_salary[
        ["title", "company.display_name", "salary_min", "salary_max"]
    ]
) 

In [ ]:
# Check important text columns for empty or whitespace-only values
text_columns = [
    "title",
    "company.display_name",
    "location.display_name",
    "description"
]

for column in text_columns:
    blank_count = (
        df_clean[column]
        .astype("string")
        .str.strip()
        .eq("")
        .sum()
    )

    print(f"{column}: {blank_count} blank values")

In [ ]:
# Count job postings by year
df_clean["posting_year"] = df_clean["created"].dt.year

year_counts = (
    df_clean["posting_year"]
    .value_counts()
    .sort_index()
)

print("Job postings by year:")
display(year_counts)

In [ ]:
# Count how many times each job title appears
title_counts = df_clean["title"].value_counts()

# Show titles that appear more than once
repeated_titles = title_counts[title_counts > 1]

print("Number of distinct job titles:", df_clean["title"].nunique())
print("Number of titles appearing more than once:", len(repeated_titles))

display(repeated_titles.head(10))

In [ ]:
# Inspect the last 120 characters of five job descriptions
for index, description in df_clean["description"].head(5).items():
    print(f"\nJob {index}:")
    print(description[-120:])
    

In [ ]:
from pathlib import Path
import json

raw_folder = Path("../data/raw")

for file_path in raw_folder.glob("*.json"):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"\nFile: {file_path.name}")
    print("Records:", len(data.get("results", [])))
    print("API reported count:", data.get("count"))

In [ ]:
response = requests.get(...)

In [ ]:
requests.get(...)

In [ ]:
requests.get(...)


In [ ]:
import os
import json
import requests
from pathlib import Path
from dotenv import load_dotenv

# Load your existing API credentials
load_dotenv("../.env")

app_id = os.getenv("ADZUNA_APP_ID")
app_key = os.getenv("ADZUNA_APP_KEY")

# Save the new data in our existing raw-data folder
raw_folder = Path("../data/raw")
raw_folder.mkdir(parents=True, exist_ok=True)

# Request one page of AI/ML Engineer jobs in India
url = "https://api.adzuna.com/v1/api/jobs/in/search/1"

params = {
    "app_id": app_id,
    "app_key": app_key,
    "what": "AI ML Engineer",
    "results_per_page": 50,
    "content-type": "application/json"
}

response = requests.get(url, params=params, timeout=30)

print("Status code:", response.status_code)

if response.status_code == 200:
    data = response.json()

    output_path = raw_folder / "adzuna_ai_ml_engineer_p1.json"

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print("Jobs returned:", len(data.get("results", [])))
    print("Saved to:", output_path)
else:
    print("Request failed. Response:", response.text[:300])

In [ ]:
params = {
    "app_id": app_id,
    "app_key": app_key,
    "what": "Data Engineer",
    "results_per_page": 50,
    "content-type": "application/json"
}

url = "https://api.adzuna.com/v1/api/jobs/in/search/1"

response = requests.get(url, params=params, timeout=30)

print("Status code:", response.status_code)

if response.status_code == 200:
    data = response.json()

    output_path = raw_folder / "adzuna_data_engineer_p1.json"

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print("Jobs returned:", len(data.get("results", [])))
    print("Saved to:", output_path)
else:
    print("Request failed:", response.text[:300])

In [ ]:
import json
import requests
from pathlib import Path

# Five job roles to collect
job_roles = {
    "business_analyst": "Business Analyst",
    "power_bi_developer": "Power BI Developer",
    "data_scientist": "Data Scientist",
    "ai_engineer": "AI Engineer",
    "ml_engineer": "Machine Learning Engineer"
}

url = "https://api.adzuna.com/v1/api/jobs/in/search/1"
results_summary = []

for file_prefix, role in job_roles.items():
    print(f"\nSearching for: {role}")

    params = {
        "app_id": app_id,
        "app_key": app_key,
        "what": role,
        "results_per_page": 50,
        "content-type": "application/json"
    }

    try:
        response = requests.get(url, params=params, timeout=30)
        print("Status code:", response.status_code)

        if response.status_code == 200:
            data = response.json()
            output_path = raw_folder / f"adzuna_{file_prefix}_p1.json"

            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)

            count = len(data.get("results", []))
            print("Jobs returned:", count)
            print("Saved to:", output_path)

            results_summary.append({
                "role": role,
                "status": 200,
                "jobs": count
            })
        else:
            print("Request failed:", response.text[:200])
            results_summary.append({
                "role": role,
                "status": response.status_code,
                "jobs": 0
            })

    except requests.RequestException as error:
        print("Request error:", error)
        results_summary.append({
            "role": role,
            "status": "Request error",
            "jobs": 0
        })

print("\n--- Collection Summary ---")
for item in results_summary:
    print(item)

In [ ]:

import json
from pathlib import Path

file_path = Path("../data/raw/adzuna_ai_engineer_p1.json")

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

job = data["results"][0]

print("Job Title:", job.get("title"))
print("Description Length:", len(job.get("description", "")))
print("Description Ending:", job.get("description", "")[-150:])
print("Job URL:", job.get("redirect_url"))


In [ ]:

import os
import json
import requests
from pathlib import Path
from dotenv import load_dotenv

# Load the API credentials from the project root
load_dotenv("../.env")

app_id = os.getenv("ADZUNA_APP_ID")
app_key = os.getenv("ADZUNA_APP_KEY")

if not app_id or not app_key:
    raise ValueError("Adzuna credentials not found in .env")

# Search for Generative AI Engineer jobs in India
url = "https://api.adzuna.com/v1/api/jobs/in/search/1"

params = {
    "app_id": app_id,
    "app_key": app_key,
    "what": "Generative AI Engineer",
    "where": "India",
    "results_per_page": 50,
    "content-type": "application/json"
}

response = requests.get(url, params=params, timeout=30)
print("Status code:", response.status_code)

response.raise_for_status()
data = response.json()

# Save the raw response without overwriting existing files
output_path = Path("../data/raw/adzuna_generative_ai_engineer_p1.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

jobs = data.get("results", [])

print("Jobs loaded:", len(jobs))
print("Unique IDs:", len({job.get("id") for job in jobs}))
print("Saved to:", output_path)

for job in jobs[:10]:
    print("-", job.get("title"), "|", job.get("company", {}).get("display_name"))


In [ ]:

import os
import json
import requests
from pathlib import Path
from dotenv import load_dotenv

load_dotenv("../.env")

url = "https://api.adzuna.com/v1/api/jobs/in/search/3"

params = {
    "app_id": os.getenv("ADZUNA_APP_ID"),
    "app_key": os.getenv("ADZUNA_APP_KEY"),
    "what": "Data Analyst",
    "where": "India",
    "results_per_page": 50,
    "content-type": "application/json"
}

response = requests.get(url, params=params, timeout=30)
print("Status code:", response.status_code)

response.raise_for_status()
data = response.json()

output_path = Path("../data/raw/adzuna_data_analyst_p3.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

jobs = data.get("results", [])

print("Jobs loaded:", len(jobs))
print("Unique IDs:", len({job.get("id") for job in jobs}))
print("Saved to:", output_path)


In [ ]:

import os
import json
import requests
from pathlib import Path
from dotenv import load_dotenv

load_dotenv("../.env")

app_id = os.getenv("ADZUNA_APP_ID")
app_key = os.getenv("ADZUNA_APP_KEY")

if not app_id or not app_key:
    raise ValueError("Adzuna credentials not found in .env")

raw_folder = Path("../data/raw")
raw_folder.mkdir(parents=True, exist_ok=True)

job_roles = {
    "bi_analyst": "BI Analyst",
    "sql_analyst": "SQL Analyst",
    "mis_analyst": "MIS Analyst",
    "junior_data_analyst": "Junior Data Analyst"
}

for file_name, keyword in job_roles.items():
    print(f"\nSearching: {keyword}")

    url = "https://api.adzuna.com/v1/api/jobs/in/search/1"

    params = {
        "app_id": app_id,
        "app_key": app_key,
        "what": keyword,
        "where": "India",
        "results_per_page": 50,
        "content-type": "application/json"
    }

    try:
        response = requests.get(url, params=params, timeout=30)
        print("Status code:", response.status_code)

        response.raise_for_status()
        data = response.json()

        output_path = raw_folder / f"adzuna_{file_name}_p1.json"

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        jobs = data.get("results", [])
        print("Jobs loaded:", len(jobs))
        print("Unique IDs:", len({
            job.get("id") for job in jobs if job.get("id")
        }))
        print("Saved:", output_path)

    except requests.RequestException as e:
        print("Request failed:", e)
        print("Stopping to avoid further API requests.")
        break


In [ ]:

import os
import json
import requests
from pathlib import Path
from dotenv import load_dotenv

load_dotenv("../.env")

url = "https://api.adzuna.com/v1/api/jobs/in/search/1"

params = {
    "app_id": os.getenv("ADZUNA_APP_ID"),
    "app_key": os.getenv("ADZUNA_APP_KEY"),
    "what": "Business Intelligence Analyst",
    "where": "India",
    "results_per_page": 50,
    "content-type": "application/json"
}

response = requests.get(url, params=params, timeout=30)
print("Status code:", response.status_code)

response.raise_for_status()
data = response.json()

output_path = Path("../data/raw/adzuna_business_intelligence_analyst_p1.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

jobs = data.get("results", [])

print("Jobs loaded:", len(jobs))
print("Unique IDs:", len({job.get("id") for job in jobs if job.get("id")}))
print("Saved to:", output_path)


In [ ]:

import os
import json
import requests
from pathlib import Path
from dotenv import load_dotenv

load_dotenv("../.env")

app_id = os.getenv("ADZUNA_APP_ID")
app_key = os.getenv("ADZUNA_APP_KEY")

if not app_id or not app_key:
    raise ValueError("Adzuna credentials not found in .env")

raw_folder = Path("../data/raw")
raw_folder.mkdir(parents=True, exist_ok=True)

roles = {
    "sql_analyst": "SQL Analyst",
    "mis_analyst": "MIS Analyst",
    "junior_data_analyst": "Junior Data Analyst"
}

for file_name, keyword in roles.items():
    print(f"\nSearching: {keyword}")

    url = "https://api.adzuna.com/v1/api/jobs/in/search/1"
    params = {
        "app_id": app_id,
        "app_key": app_key,
        "what": keyword,
        "where": "India",
        "results_per_page": 50,
        "content-type": "application/json"
    }

    try:
        response = requests.get(url, params=params, timeout=30)
        print("Status:", response.status_code)
        response.raise_for_status()

        data = response.json()
        output_path = raw_folder / f"adzuna_{file_name}_p1.json"

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        jobs = data.get("results", [])
        print("Jobs loaded:", len(jobs))
        print("Saved:", output_path)

    except requests.RequestException as e:
        print("Request failed:", e)
        print("Stopped to avoid further requests.")
        break


In [ ]:

from pathlib import Path

# Locate the folder containing our collected job data
raw_folder = Path("../data/raw")

# Find all JSON files in that folder
json_files = sorted(raw_folder.glob("*.json"))

# Display the total number of files
print("Total JSON files:", len(json_files))

# Display each filename
print("\nAvailable files:")

for file in json_files:
    print(file.name)


In [ ]:

import json
import pandas as pd
from pathlib import Path

# 1. Locate the raw JSON files
raw_folder = Path("../data/raw")
json_files = sorted(raw_folder.glob("*.json"))

# 2. Map each file to its search role
role_mapping = {
    "adzuna_ai_engineer_p1.json": "AI Engineer",
    "adzuna_ai_ml_engineer_p1.json": "AI/ML Engineer",
    "adzuna_bi_analyst_p1.json": "BI Analyst",
    "adzuna_business_analyst_p1.json": "Business Analyst",
    "adzuna_business_intelligence_analyst_p1.json":
        "Business Intelligence Analyst",
    "adzuna_data_analyst_p2.json": "Data Analyst",
    "adzuna_data_analyst_p3.json": "Data Analyst",
    "adzuna_page1.json": "Data Analyst",
    "adzuna_data_engineer_p1.json": "Data Engineer",
    "adzuna_data_scientist_p1.json": "Data Scientist",
    "adzuna_generative_ai_engineer_p1.json": "Generative AI Engineer",
    "adzuna_junior_data_analyst_p1.json": "Junior Data Analyst",
    "adzuna_mis_analyst_p1.json": "MIS Analyst",
    "adzuna_ml_engineer_p1.json": "Machine Learning Engineer",
    "adzuna_power_bi_developer_p1.json": "Power BI Developer",
    "adzuna_sql_analyst_p1.json": "SQL Analyst"
}

# 3. Read each file and collect its job records
all_jobs = []

for file_path in json_files:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    jobs = data.get("results", [])
    search_role = role_mapping.get(file_path.name, "Unknown")

    for job in jobs:
        job_record = job.copy()
        job_record["search_role"] = search_role
        job_record["source_file"] = file_path.name
        all_jobs.append(job_record)

# 4. Convert all records into one DataFrame
df_jobs = pd.json_normalize(all_jobs)

# 5. Display initial validation results
print("Files processed:", len(json_files))
print("Total job records:", len(df_jobs))
print("Total columns:", len(df_jobs.columns))

print("\nRecords by search role:")
print(df_jobs["search_role"].value_counts())

print("\nUnique job IDs:", df_jobs["id"].nunique())
print("Duplicate job IDs:", df_jobs["id"].duplicated().sum())


In [ ]:

# Find every row whose job ID appears more than once
duplicate_jobs = df_jobs[
    df_jobs["id"].duplicated(keep=False)
].sort_values("id")

print("Rows involved in repeated IDs:", len(duplicate_jobs))

# Display the important columns for investigation
columns_to_check = [
    "id",
    "title",
    "company.display_name",
    "search_role",
    "source_file"
]

print(
    duplicate_jobs[columns_to_check].to_string(index=False)
)


In [ ]:

# Count how many times each job ID appears
id_counts = df_jobs["id"].value_counts()

# IDs that appear more than once
repeated_ids = id_counts[id_counts > 1]

print("Total records:", len(df_jobs))
print("Unique non-null job IDs:", df_jobs["id"].nunique())
print("Job IDs appearing more than once:", len(repeated_ids))
print("Rows involved in repeats:",
      df_jobs["id"].isin(repeated_ids.index).sum())
print("Extra duplicate rows:",
      int((repeated_ids - 1).sum()))
print("Missing job IDs:", df_jobs["id"].isna().sum())


In [ ]:

# 1. Summarize all search categories for each job ID
role_summary = (
    df_jobs.groupby("id")["search_role"]
    .agg(lambda roles: " | ".join(sorted(set(roles.dropna()))))
)

# 2. Summarize all source files for each job ID
source_summary = (
    df_jobs.groupby("id")["source_file"]
    .agg(lambda files: " | ".join(sorted(set(files.dropna()))))
)

# 3. Keep one representative row for each job ID
df_clean = df_jobs.drop_duplicates(
    subset="id",
    keep="first"
).copy()

# 4. Preserve the full search and source history
df_clean["matched_search_roles"] = (
    df_clean["id"].map(role_summary)
)

df_clean["all_source_files"] = (
    df_clean["id"].map(source_summary)
)

# 5. Validate the cleaned dataset
print("Original records:", len(df_jobs))
print("Cleaned records:", len(df_clean))
print("Unique IDs after cleaning:", df_clean["id"].nunique())
print("Duplicate IDs remaining:",
      df_clean["id"].duplicated().sum())

print("\nSample of preserved search categories:")
print(
    df_clean[
        ["id", "title", "matched_search_roles"]
    ].head(10).to_string(index=False)
)


In [ ]:

# STEP 7: Data Quality Report

# 1. Check dataset size
print("DATASET OVERVIEW")
print("----------------")
print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))

# 2. Check missing values
print("\nMISSING VALUES")
print("--------------")
missing_report = pd.DataFrame({
    "missing_count": df_clean.isna().sum(),
    "missing_percent": (
        df_clean.isna().mean() * 100
    ).round(2)
})

print(
    missing_report[
        missing_report["missing_count"] > 0
    ].sort_values(
        "missing_count", ascending=False
    ).to_string()
)

# 3. Check important text columns
print("\nBLANK TEXT VALUES")
print("-----------------")
text_columns = [
    "title",
    "company.display_name",
    "location.display_name",
    "description"
]

for col in text_columns:
    if col in df_clean.columns:
        blank_count = (
            df_clean[col]
            .astype("string")
            .str.strip()
            .eq("")
            .fillna(False)
            .sum()
        )
        print(f"{col}: {blank_count}")

# 4. Check data types
print("\nDATA TYPES")
print("----------")
print(df_clean.dtypes.to_string())

# 5. Check salary availability
print("\nSALARY QUALITY")
print("--------------")
salary_min = "salary_min"
salary_max = "salary_max"

if salary_min in df_clean.columns and salary_max in df_clean.columns:
    both_salaries = df_clean[
        df_clean[salary_min].notna()
        & df_clean[salary_max].notna()
    ]

    invalid_salary = both_salaries[
        both_salaries[salary_min] > both_salaries[salary_max]
    ]

    print("Jobs with both salary values:", len(both_salaries))
    print("Jobs with minimum salary greater than maximum:",
          len(invalid_salary))

# 6. Check description lengths
print("\nDESCRIPTION LENGTH")
print("------------------")
df_clean["description_length"] = (
    df_clean["description"]
    .fillna("")
    .astype(str)
    .str.len()
)

print(df_clean["description_length"].describe())
print(
    "Descriptions with exactly 500 characters:",
    (df_clean["description_length"] == 500).sum()
)


In [ ]:

# Create a separate working copy
df_analysis = df_clean.copy()

# 1. Convert job posting dates into datetime format
df_analysis["created"] = pd.to_datetime(
    df_analysis["created"],
    errors="coerce",
    utc=True
)

# 2. Ensure salary columns are numeric
for col in ["salary_min", "salary_max"]:
    df_analysis[col] = pd.to_numeric(
        df_analysis[col],
        errors="coerce"
    )

# 3. Report data types of important columns
important_columns = [
    "id",
    "title",
    "company.display_name",
    "created",
    "salary_min",
    "salary_max",
    "description_length"
]

print("IMPORTANT COLUMN TYPES")
print(df_analysis[important_columns].dtypes)

# 4. Check invalid or missing posting dates
print("\nPOSTING DATE CHECK")
print("Missing or invalid dates:",
      df_analysis["created"].isna().sum())

print("Earliest posting:",
      df_analysis["created"].min())

print("Latest posting:",
      df_analysis["created"].max())

# 5. Check salary data after conversion
print("\nSALARY CHECK")
print(df_analysis[["salary_min", "salary_max"]].describe())

# 6. Check for impossible salary ranges
invalid_ranges = df_analysis[
    df_analysis["salary_min"].notna()
    & df_analysis["salary_max"].notna()
    & (df_analysis["salary_min"] > df_analysis["salary_max"])
]

print("Invalid salary ranges:", len(invalid_ranges))


In [ ]:

# Step 9: Investigate suspicious salary values

# Check how many records have each salary value missing
print("SALARY COMPLETENESS")
print("Both salaries available:",
      df_analysis[["salary_min", "salary_max"]].notna().all(axis=1).sum())

print("Minimum salary is zero:",
      (df_analysis["salary_min"] == 0).sum())

print("Maximum salary is below 10,000:",
      ((df_analysis["salary_max"] < 10000)
       & df_analysis["salary_max"].notna()).sum())

# Display listings with suspicious salary values
suspicious_salary = df_analysis[
    (
        (df_analysis["salary_min"] == 0)
        | (
            (df_analysis["salary_max"] < 10000)
            & df_analysis["salary_max"].notna()
        )
    )
]

columns_to_review = [
    "title",
    "company.display_name",
    "location.display_name",
    "salary_min",
    "salary_max",
    "search_role",
    "created"
]

print("\nSUSPICIOUS SALARY RECORDS")
print(
    suspicious_salary[columns_to_review]
    .to_string(index=False)
)


In [ ]:

# Step 10A: Flag salary records for review

df_analysis["salary_review"] = "Review not required"

# Flag zero minimum salaries
df_analysis.loc[
    df_analysis["salary_min"] == 0,
    "salary_review"
] = "Zero minimum salary"

# Flag unusually small maximum salaries
df_analysis.loc[
    df_analysis["salary_max"].notna()
    & (df_analysis["salary_max"] < 10000),
    "salary_review"
] = "Unusually low maximum salary"

print("SALARY REVIEW SUMMARY")
print(df_analysis["salary_review"].value_counts().to_string())

# Show every flagged record without truncating the rows
review_columns = [
    "title", "company.display_name",
    "salary_min", "salary_max", "salary_review"
]

print("\nRECORDS TO REVIEW")
print(
    df_analysis.loc[
        df_analysis["salary_review"] != "Review not required",
        review_columns
    ].to_string(index=False)
)

# Step 10B: Check how recent the job listings are

df_analysis["posting_year"] = df_analysis["created"].dt.year

print("\nLISTINGS BY POSTING YEAR")
print(
    df_analysis["posting_year"]
    .value_counts()
    .sort_index()
    .to_string()
)

cutoff_date = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=90)

recent_jobs = df_analysis[df_analysis["created"] >= cutoff_date]

print("\nRECENT LISTING CHECK")
print("Listings in the last 90 days:", len(recent_jobs))
print("Percentage of dataset:",
      round(len(recent_jobs) / len(df_analysis) * 100, 2), "%")


In [ ]:

# Step 11: Analyze recent listings by search category

# Use the same 90-day cutoff for a consistent comparison
recent_jobs = df_analysis[
    df_analysis["created"] >= cutoff_date
].copy()

print("RECENT LISTINGS BY SEARCH CATEGORY")
role_counts = (
    recent_jobs["search_role"]
    .value_counts()
    .rename_axis("search_role")
    .reset_index(name="recent_listing_count")
)

print(role_counts.to_string(index=False))

print("\nRECENT LISTINGS BY POSTING YEAR")
print(
    recent_jobs["posting_year"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nDATASET VALIDATION")
print("All cleaned unique job IDs:", df_analysis["id"].nunique())
print("Recent unique job IDs:", recent_jobs["id"].nunique())
print("Recent listings with missing search category:",
      recent_jobs["search_role"].isna().sum())


In [ ]:

# Step 12: Compare total and recent listings by search category

total_by_role = (
    df_analysis.groupby("search_role")["id"]
    .nunique()
    .rename("total_listings")
)

recent_by_role = (
    recent_jobs.groupby("search_role")["id"]
    .nunique()
    .rename("recent_listings")
)

role_summary = pd.concat(
    [total_by_role, recent_by_role],
    axis=1
).fillna(0)

role_summary["total_listings"] = (
    role_summary["total_listings"].astype(int)
)

role_summary["recent_listings"] = (
    role_summary["recent_listings"].astype(int)
)

role_summary["recent_percentage"] = (
    role_summary["recent_listings"]
    / role_summary["total_listings"]
    * 100
).round(2)

role_summary = role_summary.sort_values(
    "recent_listings",
    ascending=False
)

print("ROLE-LEVEL LISTING SUMMARY")
print(role_summary.to_string())


In [ ]:

import matplotlib.pyplot as plt

# Sort roles from lowest to highest for a readable horizontal chart
chart_data = role_summary.sort_values(
    "recent_listings",
    ascending=True
)

plt.figure(figsize=(10, 7))

plt.barh(
    chart_data.index,
    chart_data["recent_listings"]
)

plt.title("Recent Job Listings by Search Category")
plt.xlabel("Number of Unique Listings Posted in the Last 90 Days")
plt.ylabel("Search Category")

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

print("Matplotlib is ready!")
print("Version:", plt.matplotlib.__version__)

In [ ]:

import matplotlib.pyplot as plt

roles = ["Data Analyst", "AI Engineer", "BI Analyst"]
counts = [76, 41, 34]

plt.bar(roles, counts)

plt.title("Recent Job Listings")
plt.xlabel("Job Role")
plt.ylabel("Number of Listings")

plt.show()


In [ ]:
import matplotlib.pyplot as plt

skills = ["Python", "SQL", "Power BI"]
counts = [80, 50, 30]

plt.bar(skills, counts)
plt.title("Job Market Skill Analysis")
plt.xlabel("Skills")
plt.ylabel("Number of Job Listings")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

skills = ["Python", "SQL", "Power BI"]
counts = [80, 50, 30]

plt.bar(skills, counts, color="green")

plt.title("Job Market Skill Analysis")
plt.xlabel("Skills")
plt.ylabel("Number of Job Listings")

plt.show()

In [ ]:
import matplotlib.pyplot as plt

roles = ["Data Analyst", "AI Engineer", "BI Analyst"]
counts = [76, 41, 34]

plt.barh(roles, counts, color="green")

plt.title("Recent Job Listings by Role")
plt.xlabel("Number of Listings")
plt.ylabel("Job Role")

plt.show()

In [ ]:
import matplotlib.pyplot as plt

chart_data = role_summary.sort_values(
    "recent_listings",
    ascending=True
)

plt.figure(figsize=(10, 7))

plt.barh(
    chart_data.index,
    chart_data["recent_listings"]
)

plt.title("Recent Job Listings by Search Category")
plt.xlabel("Number of Listings")
plt.ylabel("Search Category")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

chart_data = role_summary.sort_values(
    "recent_listings",
    ascending=True
)

plt.figure(figsize=(10, 7))

plt.barh(
    chart_data.index,
    chart_data["recent_listings"]
)

plt.title("Recent Job Listings by Search Category")
plt.xlabel("Number of Listings")
plt.ylabel("Job Search Category")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

output_path = Path("../data/processed")
output_path.mkdir(parents=True, exist_ok=True)

df_analysis.to_csv(
    output_path / "india_jobs_cleaned.csv",
    index=False
)

print("Rows saved:", len(df_analysis))
print("File:", output_path / "india_jobs_cleaned.csv")

In [ ]:
import pandas as pd
from pathlib import Path

file_path = Path("../data/processed/india_jobs_cleaned.csv")

df_check = pd.read_csv(file_path)

print("Rows:", len(df_check))
print("Columns:", len(df_check.columns))
print(df_check.head(5))

In [ ]:
import sqlite3
from pathlib import Path

db_path = Path("../data/processed/job_market.db")

conn = sqlite3.connect(db_path)

df_check.to_sql(
    "job_listings",
    conn,
    if_exists="replace",
    index=False
)

print("Total jobs loaded:", pd.read_sql_query(
    "SELECT COUNT(*) AS total_jobs FROM job_listings",
    conn
).iloc[0, 0])

In [ ]:
query = """
SELECT
    search_role,
    COUNT(*) AS total_jobs
FROM job_listings
GROUP BY search_role
ORDER BY total_jobs DESC;
"""

role_counts_sql = pd.read_sql_query(query, conn)
role_counts_sql

In [ ]:
query = """
SELECT
    search_role,
    COUNT(*) AS recent_jobs
FROM job_listings
WHERE created >= date('now', '-90 days')
GROUP BY search_role
ORDER BY recent_jobs DESC;
"""

recent_role_counts = pd.read_sql_query(query, conn)
recent_role_counts

In [ ]:
query = """
SELECT
    COUNT(*) AS total_jobs,
    COUNT(salary_min) AS jobs_with_salary,
    COUNT(*) - COUNT(salary_min) AS jobs_without_salary
FROM job_listings;
"""

salary_availability = pd.read_sql_query(query, conn)
salary_availability

In [ ]:
query = """
SELECT
    COUNT(*) AS total_salary_records,
    SUM(CASE WHEN salary_min = 0 THEN 1 ELSE 0 END) AS zero_min_salary,
    SUM(CASE WHEN salary_max > 0
              AND salary_max < 10000
             THEN 1 ELSE 0 END) AS low_max_salary
FROM job_listings
WHERE salary_min IS NOT NULL
   OR salary_max IS NOT NULL;
"""

salary_quality = pd.read_sql_query(query, conn)
salary_quality

In [ ]:
query = """
SELECT
    contract_time,
    COUNT(*) AS total_jobs
FROM job_listings
GROUP BY contract_time
ORDER BY total_jobs DESC;
"""

contract_summary = pd.read_sql_query(query, conn)
contract_summary

In [ ]:
query = """
SELECT
    "location.display_name" AS location,
    COUNT(*) AS total_jobs
FROM job_listings
WHERE "location.display_name" IS NOT NULL
  AND TRIM("location.display_name") != ''
GROUP BY "location.display_name"
ORDER BY total_jobs DESC
LIMIT 10;
"""

top_locations = pd.read_sql_query(query, conn)
top_locations

In [ ]:
columns_query = """
PRAGMA table_info(job_listings);
"""

columns_df = pd.read_sql_query(columns_query, conn)
columns_df[["name", "type"]]

In [ ]:
query = """
SELECT
    "company.display_name" AS company,
    COUNT(*) AS total_jobs
FROM job_listings
WHERE "company.display_name" IS NOT NULL
  AND TRIM("company.display_name") != ''
GROUP BY "company.display_name"
ORDER BY total_jobs DESC
LIMIT 10;
"""

top_companies = pd.read_sql_query(query, conn)
top_companies

In [ ]:
query = """
SELECT
    title,
    COUNT(*) AS total_jobs
FROM job_listings
WHERE title IS NOT NULL
  AND TRIM(title) != ''
GROUP BY title
ORDER BY total_jobs DESC
LIMIT 10;
"""

top_titles = pd.read_sql_query(query, conn)
top_titles

In [ ]:
query = """
SELECT
    posting_year,
    COUNT(*) AS total_jobs
FROM job_listings
WHERE posting_year IS NOT NULL
GROUP BY posting_year
ORDER BY posting_year DESC;
"""

year_summary = pd.read_sql_query(query, conn)
year_summary

In [ ]:
skills = ["Python", "SQL", "Power BI", "Excel", "Machine Learning", "LLM", "RAG"]

skill_results = []

for skill in skills:
    query = """
    SELECT COUNT(*) AS job_count
    FROM job_listings
    WHERE LOWER(description) LIKE ?;
    """

    count = pd.read_sql_query(
        query, conn, params=(f"%{skill.lower()}%",)
    ).iloc[0, 0]

    skill_results.append({
        "skill": skill,
        "job_count": count
    })

skill_summary = pd.DataFrame(skill_results)
skill_summary.sort_values("job_count", ascending=False)

In [ ]:
query = """
SELECT
    search_role,
    COUNT(*) AS jobs_with_salary,
    ROUND(AVG(salary_min), 0) AS avg_min_salary,
    ROUND(AVG(salary_max), 0) AS avg_max_salary
FROM job_listings
WHERE salary_min > 0
  AND salary_max >= salary_min
GROUP BY search_role
HAVING COUNT(*) >= 3
ORDER BY avg_min_salary DESC;
"""

salary_by_role = pd.read_sql_query(query, conn)
salary_by_role

In [ ]:
query = """
SELECT
    search_role,
    COUNT(*) AS total_jobs,
    SUM(
        CASE
            WHEN created >= date('now', '-90 days')
            THEN 1
            ELSE 0
        END
    ) AS recent_jobs,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN created >= date('now', '-90 days')
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS recent_percentage
FROM job_listings
GROUP BY search_role
ORDER BY recent_jobs DESC;
"""

final_role_summary = pd.read_sql_query(query, conn)
final_role_summary

In [5]:

# Create a location-aware dataset for Power BI
location_jobs = pd.read_sql_query("""
    SELECT *
    FROM job_listings
""", conn)

location_jobs.to_csv(
    "../data/processed/india_jobs_powerbi.csv",
    index=False
)

print("Rows:", len(location_jobs))
print("Columns:", location_jobs.columns.tolist())
print("Saved: data/processed/india_jobs_powerbi.csv")


NameError: name 'conn' is not defined

In [6]:

from pathlib import Path
import sqlite3
import pandas as pd

project_root = Path.cwd()

# Find the existing SQLite database
db_path = project_root / "data" / "processed" / "job_market.db"

if not db_path.exists():
    project_root = project_root.parent
    db_path = project_root / "data" / "processed" / "job_market.db"

conn = sqlite3.connect(db_path)

print("Database connected:", db_path.exists())
print("Database path:", db_path)


Database connected: True
Database path: c:\Users\HP\Documents\JOB-MARKET-INTELLIGENCE\data\processed\job_market.db


In [7]:
location_jobs = pd.read_sql_query("""
    SELECT *
    FROM job_listings
""", conn)

output_path = project_root / "data" / "processed" / "india_jobs_powerbi.csv"
location_jobs.to_csv(output_path, index=False)

print("Rows:", len(location_jobs))
print("Columns:", location_jobs.columns.tolist())
print("Saved to:", output_path)

Rows: 784
Columns: ['adref', '__CLASS__', 'id', 'title', 'contract_time', 'description', 'created', 'salary_is_predicted', 'latitude', 'longitude', 'redirect_url', 'search_role', 'source_file', 'location.area', 'location.display_name', 'location.__CLASS__', 'company.display_name', 'company.__CLASS__', 'category.label', 'category.tag', 'category.__CLASS__', 'contract_type', 'salary_max', 'salary_min', 'matched_search_roles', 'all_source_files', 'description_length', 'salary_review', 'posting_year']
Saved to: c:\Users\HP\Documents\JOB-MARKET-INTELLIGENCE\data\processed\india_jobs_powerbi.csv


In [ ]:
from pathlib import Path

output_path = Path("../data/processed")
output_path.mkdir(parents=True, exist_ok=True)

final_role_summary.to_csv(
    output_path / "final_role_summary.csv",
    index=False
)

salary_by_role.to_csv(
    output_path / "salary_by_role.csv",
    index=False
)

skill_summary.to_csv(
    output_path / "skill_summary.csv",
    index=False
)

print("Final analysis files saved successfully!")

In [8]:

import pandas as pd
from pathlib import Path

jobs = pd.read_csv(
    project_root / "data" / "processed" / "india_jobs_powerbi.csv"
)

skills = [
    "SQL", "Python", "Excel", "Power BI",
    "Machine Learning", "LLM", "RAG"
]

rows = []
for _, job in jobs.iterrows():
    text = str(job.get("title", "")) + " " + str(job.get("description", ""))
    for skill in skills:
        if skill.lower() in text.lower():
            rows.append({
                "id": job["id"],
                "location.display_name": job["location.display_name"],
                "search_role": job["search_role"],
                "skill": skill
            })

skill_jobs = pd.DataFrame(rows).drop_duplicates(
    subset=["id", "skill"]
)

out = project_root / "data" / "processed" / "skill_jobs_powerbi.csv"
skill_jobs.to_csv(out, index=False)

print("Rows:", len(skill_jobs))
print("Saved:", out)


Rows: 586
Saved: c:\Users\HP\Documents\JOB-MARKET-INTELLIGENCE\data\processed\skill_jobs_powerbi.csv


In [3]:
from pathlib import Path
import pandas as pd

project_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_path = project_dir / "data" / "processed" / "india_jobs_cleaned.csv"

jobs = pd.read_csv(data_path)

print("Shape:", jobs.shape)
print("Columns:", jobs.columns.tolist())

Shape: (784, 29)
Columns: ['adref', '__CLASS__', 'id', 'title', 'contract_time', 'description', 'created', 'salary_is_predicted', 'latitude', 'longitude', 'redirect_url', 'search_role', 'source_file', 'location.area', 'location.display_name', 'location.__CLASS__', 'company.display_name', 'company.__CLASS__', 'category.label', 'category.tag', 'category.__CLASS__', 'contract_type', 'salary_max', 'salary_min', 'matched_search_roles', 'all_source_files', 'description_length', 'salary_review', 'posting_year']


In [4]:

import re

skill_keywords = [
    "Python", "SQL", "Excel", "Power BI", "Tableau",
    "Machine Learning", "Deep Learning", "LLM",
    "RAG", "Generative AI", "NLP", "Azure",
    "AWS", "PySpark", "Databricks", "FastAPI"
]

role_skill_rows = []

for _, row in jobs.iterrows():
    role = row["search_role"]
    description = str(row["description"]).lower()
    
    for skill in skill_keywords:
        pattern = r"\b" + re.escape(skill.lower()) + r"\b"
        if re.search(pattern, description):
            role_skill_rows.append({
                "search_role": role,
                "skill": skill,
                "job_count": 1
            })

role_skill_summary = pd.DataFrame(role_skill_rows)

role_skill_summary = (
    role_skill_summary
    .groupby(["search_role", "skill"], as_index=False)["job_count"]
    .sum()
    .sort_values(["search_role", "job_count"], ascending=[True, False])
)

output_path = project_dir / "data" / "processed" / "role_skill_summary.csv"
role_skill_summary.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Rows:", len(role_skill_summary))
print(role_skill_summary.head(10))


Saved to: c:\Users\HP\Documents\JOB-MARKET-INTELLIGENCE\data\processed\role_skill_summary.csv
Rows: 102
   search_role             skill  job_count
3  AI Engineer     Generative AI         10
4  AI Engineer               LLM         10
7  AI Engineer            Python          8
5  AI Engineer  Machine Learning          7
8  AI Engineer               RAG          4
6  AI Engineer               NLP          3
9  AI Engineer               SQL          3
0  AI Engineer               AWS          1
1  AI Engineer             Azure          1
2  AI Engineer           FastAPI          1
